In [66]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error

In [67]:
metriche_man = [
    "Faithfulness_man",
    "Answer_Relevancy_man",
    "Context_Precision_man",
    "Context_Recall_man",
    "Noise_Sensitivity_man",
    "Semantic_Similarity_man",
    "Answer_Correctness_man",
]

metriche_mod = [
    "Faithfulness_mod",
    "Answer_Relevancy_mod",
    "Context_Precision_mod",
    "Context_Recall_mod",
    "Noise_Sensitivity_mod",
    "Semantic_Similarity_mod",
    "Answer_Correctness_mod",
]

In [68]:
df_manuale = pd.read_csv("./eval_manuale.csv")
N = len(df_manuale)
df_manuale = df_manuale.rename(
        columns={
            "ID": "ID_man",
            "question": "Question_man",
            "faithfulness": "Faithfulness_man",
            "answer_relevancy": "Answer_Relevancy_man",
            "context_precision": "Context_Precision_man",
            "context_recall": "Context_Recall_man",
            "noise_sensitivity(mode=relevant)": "Noise_Sensitivity_man",
            "semantic_similarity": "Semantic_Similarity_man",
            "answer_correctness": "Answer_Correctness_man",
        }
    )
df_manuale.info()

<class 'pandas.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID_man                   48 non-null     int64  
 1   Faithfulness_man         48 non-null     float64
 2   Answer_Relevancy_man     48 non-null     float64
 3   Context_Precision_man    48 non-null     float64
 4   Context_Recall_man       48 non-null     float64
 5   Noise_Sensitivity_man    48 non-null     float64
 6   Semantic_Similarity_man  48 non-null     float64
 7   Answer_Correctness_man   48 non-null     float64
dtypes: float64(7), int64(1)
memory usage: 3.1 KB


In [69]:
df_modello = pd.read_csv("./risultati_eval_GPT4.1.csv")
df_modello = df_modello.iloc[:N]

df_modello["ID_model"] = range(1, len(df_modello) + 1)
df_modello = df_modello.rename(
        columns={
            "ID": "ID_mod",
            "question": "Question_mod",
            "faithfulness": "Faithfulness_mod",
            "answer_relevancy": "Answer_Relevancy_mod",
            "context_precision": "Context_Precision_mod",
            "context_recall": "Context_Recall_mod",
            "noise_sensitivity(mode=relevant)": "Noise_Sensitivity_mod",
            "semantic_similarity": "Semantic_Similarity_mod",
            "answer_correctness": "Answer_Correctness_mod",
        }
    )
df_modello.info()
# righe_nan = df_modello[df_modello["noise_sensitivity(mode=relevant)"].isna()]
# print(righe_nan)
# df_modello = df_modello.fillna(0.5)

<class 'pandas.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Question_mod             48 non-null     str    
 1   groundtruth              48 non-null     str    
 2   difficulty               48 non-null     str    
 3   response                 48 non-null     str    
 4   scores                   48 non-null     str    
 5   sources                  48 non-null     str    
 6   Faithfulness_mod         48 non-null     float64
 7   Answer_Relevancy_mod     48 non-null     float64
 8   Context_Precision_mod    48 non-null     float64
 9   Context_Recall_mod       48 non-null     float64
 10  Noise_Sensitivity_mod    45 non-null     float64
 11  Semantic_Similarity_mod  48 non-null     float64
 12  Answer_Correctness_mod   48 non-null     float64
 13  ID_model                 48 non-null     int64  
dtypes: float64(7), int64(1), str(6)
memory 

In [70]:
df = pd.merge(df_manuale, df_modello, left_on="ID_man", right_on="ID_model")
# print(df.head())
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID_man                   48 non-null     int64  
 1   Faithfulness_man         48 non-null     float64
 2   Answer_Relevancy_man     48 non-null     float64
 3   Context_Precision_man    48 non-null     float64
 4   Context_Recall_man       48 non-null     float64
 5   Noise_Sensitivity_man    48 non-null     float64
 6   Semantic_Similarity_man  48 non-null     float64
 7   Answer_Correctness_man   48 non-null     float64
 8   Question_mod             48 non-null     str    
 9   groundtruth              48 non-null     str    
 10  difficulty               48 non-null     str    
 11  response                 48 non-null     str    
 12  scores                   48 non-null     str    
 13  sources                  48 non-null     str    
 14  Faithfulness_mod         48 non-null   

In [71]:
colonne_chiave = metriche_man + metriche_mod
df = df.dropna(subset=colonne_chiave).copy()
df.info()

<class 'pandas.DataFrame'>
Index: 45 entries, 0 to 47
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID_man                   45 non-null     int64  
 1   Faithfulness_man         45 non-null     float64
 2   Answer_Relevancy_man     45 non-null     float64
 3   Context_Precision_man    45 non-null     float64
 4   Context_Recall_man       45 non-null     float64
 5   Noise_Sensitivity_man    45 non-null     float64
 6   Semantic_Similarity_man  45 non-null     float64
 7   Answer_Correctness_man   45 non-null     float64
 8   Question_mod             45 non-null     str    
 9   groundtruth              45 non-null     str    
 10  difficulty               45 non-null     str    
 11  response                 45 non-null     str    
 12  scores                   45 non-null     str    
 13  sources                  45 non-null     str    
 14  Faithfulness_mod         45 non-null     flo

In [72]:
df["Global_Score_man"] = df[metriche_man].mean(axis=1)

In [73]:
X_model = df[metriche_mod].values
y_human = df["Global_Score_man"].values

In [74]:
def objective_function_mae(weights):
    global_score_model = np.dot(X_model, weights)
    mae = mean_absolute_error(y_human, global_score_model)
    return mae

def objective_function_spearman(weights):
    global_score_model = np.dot(X_model, weights)
    # spearman è invertito
    corr, _ = spearmanr(y_human, global_score_model)
    if np.isnan(corr):
        corr = -1.0
    loss_spearman = 1.0 - corr
    return loss_spearman

In [75]:
constraints = {"type": "eq", "fun": lambda w: np.sum(w) - 1.0}
bounds = [(0.0, 1.0) for _ in range(len(metriche_mod))]
initial_weights = np.ones(len(metriche_mod)) / len(metriche_mod)

In [76]:
result_mae = minimize(
    objective_function_mae,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
)
result_spearman = minimize(
    objective_function_spearman,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
)

pesi_ottimi_mae = result_mae.x
pesi_ottimi_spearman = result_spearman.x

In [77]:
for metrica, peso in zip(metriche_mod, pesi_ottimi_mae):
    peso_clean = peso if peso > 1e-4 else 0.0
    print(f"{metrica}: {peso_clean*100:.2f}%")

print("-" * 45)

for metrica, peso in zip(metriche_mod, pesi_ottimi_spearman):
    peso_clean = peso if peso > 1e-4 else 0.0
    print(f"{metrica}: {peso_clean*100:.2f}%")

Faithfulness_mod: 0.00%
Answer_Relevancy_mod: 2.82%
Context_Precision_mod: 0.00%
Context_Recall_mod: 25.10%
Noise_Sensitivity_mod: 8.57%
Semantic_Similarity_mod: 2.27%
Answer_Correctness_mod: 61.24%
---------------------------------------------
Faithfulness_mod: 14.29%
Answer_Relevancy_mod: 14.29%
Context_Precision_mod: 14.29%
Context_Recall_mod: 14.29%
Noise_Sensitivity_mod: 14.29%
Semantic_Similarity_mod: 14.29%
Answer_Correctness_mod: 14.29%


In [78]:
df["Global_Score_model_ottimizzato"] = np.dot(X_model, pesi_ottimi_mae)
mae_finale = np.mean(
    np.abs(df["Global_Score_man"] - df["Global_Score_model_ottimizzato"])
)
df["Global_Score_model_ottimizzato"] = np.dot(X_model, pesi_ottimi_spearman)
spearman_finale, _ = spearmanr(
    df["Global_Score_man"], df["Global_Score_model_ottimizzato"]
)

In [79]:
print(f"MAE: {mae_finale:.4f}")
print(f"Spearman: {spearman_finale:.4f}")

MAE: 0.0811
Spearman: 0.7603
